# Lab | Data Cleaning and Formatting

In this lab, we will be working with the customer data from an insurance company, which can be found in the CSV file located at the following link: https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv


# Challenge 1: Data Cleaning and Formatting

## Exercise 1: Cleaning Column Names

To ensure consistency and ease of use, standardize the column names of the dataframe. Start by taking a first look at the dataframe and identifying any column names that need to be modified. Use appropriate naming conventions and make sure that column names are descriptive and informative.

*Hint*:
- *Column names should be in lower case*
- *White spaces in column names should be replaced by `_`*
- *`st` could be replaced for `state`*

In [ ]:
import pandas as pd
from IPython.display import display

URL = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv"
raw = pd.read_csv(URL)
display(raw.head())
print("Columnas originales:", raw.columns.tolist())

def clean_column_names(data):
    result = data.copy()
    result.columns = result.columns.str.strip().str.lower().str.replace(r"\s+", "_", regex=True)
    return result.rename(columns={"st": "state"})

df = clean_column_names(raw)
print("Columnas normalizadas:", df.columns.tolist())

## Exercise 2: Cleaning invalid Values

The dataset contains columns with inconsistent and incorrect values that could affect the accuracy of our analysis. Therefore, we need to clean these columns to ensure that they only contain valid data.

Note that this exercise will focus only on cleaning inconsistent values and will not involve handling null values (NaN or None).

*Hint*:
- *Gender column contains various inconsistent values such as "F", "M", "Femal", "Male", "female", which need to be standardized, for example, to "M" and "F".*
- *State abbreviations be can replaced with its full name, for example "AZ": "Arizona", "Cali": "California", "WA": "Washington"*
- *In education, "Bachelors" could be replaced by "Bachelor"*
- *In Customer Lifetime Value, delete the `%` character*
- *In vehicle class, "Sports Car", "Luxury SUV" and "Luxury Car" could be replaced by "Luxury"*

In [ ]:
def clean_invalid_values(data):
    result = data.copy()
    for column in result.select_dtypes(include=["object", "string"]):
        result[column] = result[column].astype("string").str.strip()
    result["gender"] = result["gender"].str.lower().replace({"f": "F", "female": "F", "femal": "F", "m": "M", "male": "M"})
    result["state"] = result["state"].replace({"AZ": "Arizona", "Cali": "California", "CA": "California", "WA": "Washington"})
    result["education"] = result["education"].replace({"Bachelors": "Bachelor"})
    result["customer_lifetime_value"] = result["customer_lifetime_value"].str.replace("%", "", regex=False)
    result["vehicle_class"] = result["vehicle_class"].replace({"Sports Car": "Luxury", "Luxury SUV": "Luxury", "Luxury Car": "Luxury"})
    return result

for column in ["gender", "state", "education", "vehicle_class"]:
    print(column, df[column].unique())
df = clean_invalid_values(df)
display(df.head())

## Exercise 3: Formatting data types

The data types of many columns in the dataset appear to be incorrect. This could impact the accuracy of our analysis. To ensure accurate analysis, we need to correct the data types of these columns. Please update the data types of the columns as appropriate.

It is important to note that this exercise does not involve handling null values (NaN or None).

*Hint*:
- *Customer lifetime value should be numeric*
- *Number of open complaints has an incorrect format. Look at the different values it takes with `unique()` and take the middle value. As an example, 1/5/00 should be 5. Number of open complaints is a string - remember you can use `split()` to deal with it and take the number you need. Finally, since it should be numeric, cast the column to be in its proper type.*

In [ ]:
print(df["number_of_open_complaints"].unique())

def format_types(data):
    result = data.copy()
    complaints = result["number_of_open_complaints"].astype("string")
    # Los valores 1/5/00 representan cinco quejas, no una fecha.
    complaints = complaints.map(lambda value: value.split("/")[1] if pd.notna(value) and "/" in value else value)
    result["number_of_open_complaints"] = pd.to_numeric(complaints, errors="raise").astype("Int64")
    for column in ["customer_lifetime_value", "income", "monthly_premium_auto", "total_claim_amount"]:
        result[column] = pd.to_numeric(result[column], errors="raise")
    return result

df = format_types(df)
print(df.dtypes)

## Exercise 4: Dealing with Null values

Identify any columns with null or missing values. Identify how many null values each column has. You can use the `isnull()` function in pandas to find columns with null values.

Decide on a strategy for handling the null values. There are several options, including:

- Drop the rows or columns with null values
- Fill the null values with a specific value (such as the column mean or median for numerical variables, and mode for categorical variables)
- Fill the null values with the previous or next value in the column
- Fill the null values based on a more complex algorithm or model (note: we haven't covered this yet)

Implement your chosen strategy to handle the null values. You can use the `fillna()` function in pandas to fill null values or `dropna()` function to drop null values.

Verify that your strategy has successfully handled the null values. You can use the `isnull()` function again to check if there are still null values in the dataset.

Remember to document your process and explain your reasoning for choosing a particular strategy for handling null values.

After formatting data types, as a last step, convert all the numeric variables to integers.

In [ ]:
print("Nulos antes:")
print(df.isna().sum())

def handle_nulls(data):
    # No inventar clientes a partir de filas vacias o sin identificador.
    result = data.dropna(how="all").dropna(subset=["customer"]).copy()
    # Deduplicar antes de calcular medianas evita ponderar registros repetidos.
    reference = result.drop_duplicates()
    numeric = result.select_dtypes(include="number").columns
    for column in numeric:
        result[column] = result[column].fillna(reference[column].median())
    # Unknown conserva la incertidumbre: no inferimos genero ni estado por la moda.
    for column in result.select_dtypes(include=["object", "string"]):
        result[column] = result[column].fillna("Unknown")
    # El enunciado pide enteros: redondeamos; se pierde precision monetaria.
    result[numeric] = result[numeric].round().astype("int64")
    return result

df = handle_nulls(df)
print("Nulos despues:")
print(df.isna().sum())
assert not df.isna().any().any()

## Exercise 5: Dealing with duplicates

Use the `.duplicated()` method to identify any duplicate rows in the dataframe.

Decide on a strategy for handling the duplicates. Options include:
- Dropping all duplicate rows
- Keeping only the first occurrence of each duplicated row
- Keeping only the last occurrence of each duplicated row
- Dropping duplicates based on a subset of columns
- Dropping duplicates based on a specific column

Implement your chosen strategy using the `drop_duplicates()` function.

Verify that your strategy has successfully handled the duplicates by checking for duplicates again using `.duplicated()`.

Remember to document your process and explain your reasoning for choosing a particular strategy for handling duplicates.

Save the cleaned dataset to a new CSV file.

*Hint*: *after dropping duplicates, reset the index to ensure consistency*.

In [ ]:
print("Duplicados exactos:", df.duplicated().sum())

def remove_duplicates(data):
    # Solo duplicados exactos; dos registros distintos del mismo cliente se conservan.
    return data.drop_duplicates(keep="first").reset_index(drop=True)

df = remove_duplicates(df)
assert not df.duplicated().any()
df.to_csv("customers_cleaned.csv", index=False)
print("Filas limpias:", len(df))
display(df.head())

# Bonus: Challenge 2: creating functions on a separate `py` file

Put all the data cleaning and formatting steps into functions, and create a main function that performs all the cleaning and formatting.

Write these functions in separate .py file(s). By putting these steps into functions, we can make the code more modular and easier to maintain.

*Hint: autoreload module is a utility module in Python that allows you to automatically reload modules in the current session when changes are made to the source code. This can be useful in situations where you are actively developing code and want to see the effects of changes you make without having to constantly restart the Python interpreter or Jupyter Notebook kernel.*

In [ ]:
from pathlib import Path

# Generamos el modulo reutilizable en un archivo .py separado.
Path("cleaning.py").write_text("import pandas as pd\n\ndef clean_column_names(data):\n    result = data.copy()\n    result.columns = result.columns.str.strip().str.lower().str.replace(r\"\\s+\", \"_\", regex=True)\n    return result.rename(columns={\"st\": \"state\"})\n\ndef clean_invalid_values(data):\n    result = data.copy()\n    for column in result.select_dtypes(include=[\"object\", \"string\"]):\n        result[column] = result[column].astype(\"string\").str.strip()\n    result[\"gender\"] = result[\"gender\"].str.lower().replace({\"f\": \"F\", \"female\": \"F\", \"femal\": \"F\", \"m\": \"M\", \"male\": \"M\"})\n    result[\"state\"] = result[\"state\"].replace({\"AZ\": \"Arizona\", \"Cali\": \"California\", \"CA\": \"California\", \"WA\": \"Washington\"})\n    result[\"education\"] = result[\"education\"].replace({\"Bachelors\": \"Bachelor\"})\n    result[\"customer_lifetime_value\"] = result[\"customer_lifetime_value\"].str.replace(\"%\", \"\", regex=False)\n    result[\"vehicle_class\"] = result[\"vehicle_class\"].replace({\"Sports Car\": \"Luxury\", \"Luxury SUV\": \"Luxury\", \"Luxury Car\": \"Luxury\"})\n    return result\n\ndef format_types(data):\n    result = data.copy()\n    complaints = result[\"number_of_open_complaints\"].astype(\"string\")\n    # Los valores 1/5/00 representan cinco quejas, no una fecha.\n    complaints = complaints.map(lambda value: value.split(\"/\")[1] if pd.notna(value) and \"/\" in value else value)\n    result[\"number_of_open_complaints\"] = pd.to_numeric(complaints, errors=\"raise\").astype(\"Int64\")\n    for column in [\"customer_lifetime_value\", \"income\", \"monthly_premium_auto\", \"total_claim_amount\"]:\n        result[column] = pd.to_numeric(result[column], errors=\"raise\")\n    return result\n\ndef handle_nulls(data):\n    # No inventar clientes a partir de filas vacias o sin identificador.\n    result = data.dropna(how=\"all\").dropna(subset=[\"customer\"]).copy()\n    # Deduplicar antes de calcular medianas evita ponderar registros repetidos.\n    reference = result.drop_duplicates()\n    numeric = result.select_dtypes(include=\"number\").columns\n    for column in numeric:\n        result[column] = result[column].fillna(reference[column].median())\n    # Unknown conserva la incertidumbre: no inferimos genero ni estado por la moda.\n    for column in result.select_dtypes(include=[\"object\", \"string\"]):\n        result[column] = result[column].fillna(\"Unknown\")\n    # El enunciado pide enteros: redondeamos; se pierde precision monetaria.\n    result[numeric] = result[numeric].round().astype(\"int64\")\n    return result\n\ndef remove_duplicates(data):\n    # Solo duplicados exactos; dos registros distintos del mismo cliente se conservan.\n    return data.drop_duplicates(keep=\"first\").reset_index(drop=True)\n\ndef clean_data(data):\n    result = clean_column_names(data)\n    result = clean_invalid_values(result)\n    result = format_types(result)\n    result = handle_nulls(result)\n    return remove_duplicates(result)\n", encoding="utf-8")
from cleaning import clean_data
pd.testing.assert_frame_equal(df, clean_data(raw))
print("Modulo cleaning.py generado; resultado equivalente al notebook.")

# Bonus: Challenge 3: Analyzing Clean and Formated Data

You have been tasked with analyzing the data to identify potential areas for improving customer retention and profitability. Your goal is to identify customers with a high policy claim amount and a low customer lifetime value.

In the Pandas Lab, we only looked at high policy claim amounts because we couldn't look into low customer lifetime values. If we had tried to work with that column, we wouldn't have been able to because customer lifetime value wasn't clean and in its proper format. So after cleaning and formatting the data, let's get some more interesting insights!

Instructions:

- Review the statistics again for total claim amount and customer lifetime value to gain an understanding of the data.
- To identify potential areas for improving customer retention and profitability, we want to focus on customers with a high policy claim amount and a low customer lifetime value. Consider customers with a high policy claim amount to be those in the top 25% of the total claim amount, and clients with a low customer lifetime value to be those in the bottom 25% of the customer lifetime value. Create a pandas DataFrame object that contains information about customers with a policy claim amount greater than the 75th percentile and a customer lifetime value in the bottom 25th percentile.
- Use DataFrame methods to calculate summary statistics about the high policy claim amount and low customer lifetime value data. To do so, select both columns of the dataframe simultaneously and pass it to the `.describe()` method. This will give you descriptive statistics, such as mean, median, standard deviation, minimum and maximum values for both columns at the same time, allowing you to compare and analyze their characteristics.

In [ ]:
columns = ["total_claim_amount", "customer_lifetime_value"]
display(df[columns].describe())
claim_p75 = df["total_claim_amount"].quantile(0.75)
clv_p25 = df["customer_lifetime_value"].quantile(0.25)
high_claim_low_value = df.loc[(df["total_claim_amount"] > claim_p75) & (df["customer_lifetime_value"] <= clv_p25)].copy()
print("Umbrales:", claim_p75, clv_p25)
print("Clientes seleccionados:", len(high_claim_low_value))
display(high_claim_low_value)
display(high_claim_low_value[columns].describe())
# Este filtro identifica un segmento para estudiar, no demuestra falta de rentabilidad.

Notas: las medianas se calculan sin duplicados exactos y despues de excluir filas vacias o sin cliente. Unknown evita asignar categorias personales sin evidencia. Los importes se redondean a enteros solo por exigencia del ejercicio. El porcentaje de CLV se elimina sin dividir entre 100, segun el enunciado. Solucion preparada con asistencia de IA; las salidas no se han fabricado y requieren ejecutar el cuaderno.